# Very simple comparison: Greedy Best-First vs. real DSD

Change only the **INPUTS** cell, then run all cells from top to bottom. Supply a one-dimensional population vector $y$ and its first-order inclusion-probability vector $\pi$.

Both methods use the same population values $y_k$ and first-order inclusion probabilities $\pi_k$. The quantity compared is the exact design variance of the Narain--Horvitz--Thompson total $\hat t_y=\sum_{k\in s}y_k/\pi_k$.

In [ ]:
# INPUTS: this is the only cell you normally edit.
import numpy as np
import pandas as pd

# One-dimensional finite population: replace these y values with yours.
population = np.array([12.0, 18.0, 10.0, 25.0, 14.0, 30.0, 22.0, 35.0])

# First-order inclusion probabilities. Their sum must be an integer n.
first_order = np.array([0.25, 0.35, 0.30, 0.45, 0.40, 0.50, 0.35, 0.40])

MAX_ITERATIONS = 500
M = 50                 # print efficiency after every M iterations
NEIGHBORS = 30         # candidate swaps generated per iteration
MAX_OPEN_SIZE = 500    # maximum search-frontier size
SEED = 123


In [ ]:
# Validate and prepare the population.
from graphical_sampling.population import Population
from graphical_sampling.design import Design
from graphical_sampling.order import Order

y = np.asarray(population, dtype=float).reshape(-1)
pi = np.asarray(first_order, dtype=float).reshape(-1)

if len(y) != len(pi):
    raise ValueError('population and first_order must have the same length.')
if not np.all(np.isfinite(y)) or not np.all(np.isfinite(pi)):
    raise ValueError('y and first_order must contain only finite values.')
if np.any(pi <= 0) or np.any(pi >= 1):
    raise ValueError('For this DSD construction every pi must satisfy 0 < pi < 1.')
n = int(round(pi.sum()))
if n < 1 or n >= len(pi) or not np.isclose(pi.sum(), n, atol=1e-9):
    raise ValueError('sum(first_order) must be an integer n with 1 <= n < N.')

# Population requires (N, 2) coordinates internally. These placeholders merely
# encode the one-dimensional row order; no spatial coordinates are needed.
coords = np.column_stack((np.arange(len(y), dtype=float), np.zeros(len(y))))

pop = Population(coords=coords, inclusions=pi, variable=y)
print(f'Population size N = {len(y)}; fixed sample size n = {n}')

## 1. Your package’s Greedy Best-First Search

This section directly uses `graphical_sampling.search.GreedyBestFirstSearch` with `VarNHT`. The initial graphical design is flattened to one cluster/zone so the package GBFS can generate order-swap neighbors.


In [ ]:
import random
import importlib
from graphical_sampling.criteria import VarNHT
import graphical_sampling.search.gbfs as gbfs_module

# Reload so an already-running Jupyter kernel picks up the current GBFS fixes.
gbfs_module = importlib.reload(gbfs_module)
GreedyBestFirstSearch = gbfs_module.GreedyBestFirstSearch

random.seed(SEED)
np.random.seed(SEED)

# The package GBFS changes the graphical order while preserving first-order pi.
initial_design = Design(pop).flatten()
baseline_variance = float(initial_design.nht_variance)
criteria = VarNHT()
gbfs = GreedyBestFirstSearch([initial_design], criteria)

progress_rows = [{
    'iteration': 0,
    'variance': baseline_variance,
    'efficiency_gain_pct': 0.0,
}]

def report_every_M(search, iteration):
    if iteration % M == 0 or iteration == MAX_ITERATIONS:
        best_variance = float(search.reservoir_val)
        gain = 100.0 * (baseline_variance - best_variance) / baseline_variance
        progress_rows.append({'iteration': iteration, 'variance': best_variance,
                              'efficiency_gain_pct': gain})
        print(f'[M REPORT] iteration {iteration:>6} | best variance = {best_variance:.6g} '
              f'| efficiency gain = {gain:.2f}%')

gbfs.run(
    max_iterations=MAX_ITERATIONS,
    max_open_set_size=MAX_OPEN_SIZE,
    top_k=1,
    num_new_order_nodes=NEIGHBORS,
    num_new_exchange_nodes=0,
    num_clusters_range=(1, 1),
    num_zones_range=(1, 1),
    num_changes_range=(1, 1),
    num_zone_changes=0,
    pull_strategy='default',
    exchange_coef=0.75,
    num_explore=1,
    window=None,
    n_jobs=1,
    backtrack_depth=1,
    stagnation_limit=MAX_ITERATIONS + 1,
    output_details=False,
    progress_callback=report_every_M,
)

gbfs_best_design = gbfs.best_design
gbfs_variance = float(gbfs_best_design.nht_variance)
gbfs_history = pd.DataFrame(progress_rows)


## 2. Sort by $y_k / \pi_k$ and build the real DSD Hermitian matrix

For a real DSD, $V$ has orthonormal columns and $K=VV^T$. Thus $K$ is a real symmetric (therefore Hermitian) projection kernel, $K^2=K$, and $\operatorname{diag}(K)=\pi$. The sorting is descending; change `ascending=False` below if you need ascending order.

In [ ]:
def real_dsd_factor(pi):
    """Construct the real N x n DSD factor V from ordered first-order pi."""
    pi = np.asarray(pi, dtype=float).reshape(-1)
    N = len(pi)
    n = int(round(pi.sum()))
    cumulative = 0.0
    previous_integer = 0
    threshold = 1
    s = np.zeros(N)
    crossing_rows = []

    for k, probability in enumerate(pi):
        before = cumulative
        cumulative += probability
        if cumulative >= threshold - 1e-12:
            alpha = threshold - before
            s[k] = np.sqrt(max(0.0, (1.0 - probability) / (1.0 - alpha)))
            crossing_rows.append(k)
            previous_integer = threshold
            threshold += 1
        else:
            s[k] = np.sqrt(probability / (previous_integer + 1.0 - before))
        s[k] = np.clip(s[k], 0.0, 1.0)

    V = np.zeros((N, n), dtype=float)
    V[0, 0] = 1.0
    for column in range(1, n):
        V[crossing_rows[column - 1] + 1, column] = 1.0

    c = np.sqrt(np.maximum(0.0, 1.0 - s * s))
    for k in range(N - 1):
        upper, lower = V[k].copy(), V[k + 1].copy()
        V[k] = s[k] * upper - c[k] * lower
        V[k + 1] = c[k] * upper + s[k] * lower
    return V

ratio = y / pi
sort_index = np.argsort(-ratio, kind='stable')
sorted_population = pd.DataFrame({
    'original_index': sort_index,
    'y': y[sort_index],
    'pi': pi[sort_index],
    'y_over_pi': ratio[sort_index],
})

pi_sorted = pi[sort_index]
y_sorted = y[sort_index]
V = real_dsd_factor(pi_sorted)
K = V @ V.T  # optimized real-valued Hermitian matrix

checks = pd.Series({
    'max |K - K.T|': np.max(np.abs(K - K.T)),
    'max |K @ K - K|': np.max(np.abs(K @ K - K)),
    'max |diag(K) - sorted pi|': np.max(np.abs(np.diag(K) - pi_sorted)),
    'trace(K)': np.trace(K),
})
display(sorted_population)
display(checks.to_frame('value'))

## 3. Exact DSD variance and comparison

For the projection DSD kernel, $\operatorname{Cov}(I)=\operatorname{diag}(\pi)-K\circ K$ in the real case. The exact NHT variance is therefore $(y/\pi)^T\operatorname{Cov}(I)(y/\pi)$.

In [ ]:
def real_dsd_nht_variance(y, pi, K):
    weights = np.asarray(y, float) / np.asarray(pi, float)
    covariance = np.diag(pi) - K * K  # elementwise square; real-valued fast path
    variance = float(weights @ covariance @ weights)
    return max(0.0, variance) if variance > -1e-8 else variance

dsd_variance = real_dsd_nht_variance(y_sorted, pi_sorted, K)

comparison = pd.DataFrame({
    'method': ['Initial graphical order', 'Greedy best-first', 'Real DSD (sorted y/pi)'],
    'exact_NHT_variance': [baseline_variance, gbfs_variance, dsd_variance],
})
comparison['efficiency_gain_vs_initial_pct'] = (
    100.0 * (baseline_variance - comparison['exact_NHT_variance']) / baseline_variance
)
comparison['relative_efficiency_vs_initial'] = (
    baseline_variance / comparison['exact_NHT_variance']
)
display(comparison.style.format({
    'exact_NHT_variance': '{:,.6g}',
    'efficiency_gain_vs_initial_pct': '{:.2f}%',
    'relative_efficiency_vs_initial': '{:.3f}x',
}))

winner = comparison.loc[comparison['exact_NHT_variance'].idxmin(), 'method']
print(f'Lower variance is better. Winner: {winner}')

In [ ]:
# Optional: visualize the greedy search progress.
ax = gbfs_history.plot(
    x='iteration', y='efficiency_gain_pct', marker='o', legend=False, grid=True,
    title=f'Greedy best-first efficiency (reported every M={M} iterations)'
)
ax.set_ylabel('Efficiency gain vs. initial design (%)');